# 01. Ingesta de datos
**Objetivo:** descargar los CSV desde https://mibici.net/es/datos-abiertos con un rango de fecha establecido, identificar diferentes esquemas y codificación entre los diferentes CSVs.

In [8]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import requests

from mi_bici import data as ds
from mi_bici.config import RAW_DATA_DIR

pd.set_option('display.max_colwidth', None)


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Descargar datos

In [14]:
# Descargar un archivo de prueba

url = sources['trips']['2024-01']

try:
    with requests.get(url, stream=True, timeout=30) as response:
        response.raise_for_status()
except requests.HTTPError as error:
    print(f'{type(error).__name__}: {error}')

HTTPError: 406 Client Error: Not Acceptable for url: https://mibici.net/site/assets/files/61450/datos_abiertos_2024_01.csv


In [15]:
# ¿Porqué no se pueden descargar los archivos?

user_agents = ['python-requests/2.32', ds.HTTP_HEADERS['User-Agent'], 'Mozilla/5.0 (X11; Linux x86_64)']

rows = []
for user_agent in user_agents:
    with requests.get(sources['stations'], headers={'User-Agent': user_agent}, stream=True, timeout=30) as response:
        rows.append({'user_agent': user_agent, 'status_code': response.status_code})

pd.DataFrame(rows)

,user_agent,status_code
0,python-requests/2.32,406
1,mi-bici-tc5061-equipo11,200
2,Mozilla/5.0 (X11; Linux x86_64),200


In [18]:
# Definir rango
START = '2024-01'
END = '2026-09'

sources = ds.load_sources()
trip_urls = ds.select_trip_urls(sources, START, END)

print(f'Meses a descargar: {len(trip_urls)}')

Meses: 33


In [24]:
# Descargar archivos
stations_path = ds.download_stations(sources)
trip_paths = ds.download_trips(trip_urls)

print(f'Catálogo de estaciones: {stations_path.name}')
print(f'Meses descargados: {len(trip_paths)}')

Catálogo de estaciones: nomenclatura_2026_09.csv
Meses descargados: 33


In [25]:
# Revisar peso de los archivos

print(f'Archivos por mes - Total: {files["MB"].sum():,.1f} MB')
print(f'Archivos por mes - Promedio: {files["MB"].mean():,.1f} MB, mínimo {files["MB"].min():,.1f}, máximo {files["MB"].max():,.1f}')
print(f'Catálogo de estaciones: {stations_path.stat().st_size / 1e3:,.1f} KB')

Archivos por mes - Total: 1,070.1 MB
Archivos por mes - Promedio: 32.4 MB, mínimo 25.4, máximo 42.6
Catálogo de estaciones: 40.8 KB


## Hallazgos
- El servidor de MiBico bloquea los requests on el user agent default de `request`
- El peso promedio de los archivos de cada año_mes es de 32.4 MB